In [1]:
import os
import csv
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from matplotlib import pyplot as plt

from tokeniser import get_tokenizer
from utils.dataset import COHWRDataset, ocollate_fn
from model.joint_multi_modality_model import JointMultiModel
from model.joint_model import JointModel
from train_uhwr_online import evaluate as evaluate_online
from train_uhwr import evaluate as evaluate_ft


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    print(f"Seed set to {seed}")


ROOT = "C:\\AliCode\\ICPR\\main_repo"
DATA_ROOT = "C:\\AliCode\\Datasets\\data_online_line_width_alpha"
os.chdir(ROOT)
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

Seed set to 42


device(type='cuda')

In [2]:
# ------------------------------------------------------------------
# Config
# ------------------------------------------------------------------
SEEDS = [42, 123, 456, 789, 2024]
ONLINE_RUNS = "runs_online_camera_ready"      # JointMultiModel (aux channels)
FINETUNE_RUNS = "runs_finetune_camera_ready"  # JointModel (stroke only)

# Decode used for both the per-epoch scan and the final ablations.
# beam_search matches the reported numbers; switch to "greedy" for a much
# faster (but slightly different) checkpoint scan.
DECODE_MODE = "beam_search"

# Checkpoint-scan window (see select_best_checkpoint):
SCAN_LAST_FRACTION = 0.40  # only scan the last 40% of epochs
SCAN_EXCLUDE_TAIL = 9      # ...and drop the final 9 epochs of the run

img_feat = "img_stroke"
aux_feat = [
    "img_dx",
    "img_dy",
    "img_sin_theta",
    "img_cos_theta",
    "img_curvature",
    "img_speed",
    "img_acceleration",
    "img_time_norm",
    "img_pressure",
    "img_x_tilt",
    "img_y_tilt",
]

eval_df = pd.read_csv(f"{DATA_ROOT}/val_leakproof.csv")
tokenizer = get_tokenizer()
len(eval_df)

253

In [3]:
# ------------------------------------------------------------------
# Build eval loaders once (val_leakproof)
# ------------------------------------------------------------------
# Online: full aux channels present; ablation zeroes channels at inference.
online_dataset = COHWRDataset(
    DATA_ROOT, eval_df, tokenizer,
    img_feat=img_feat, aux_feat=aux_feat, cache_dir="val_cache3",
)
online_loader = DataLoader(
    online_dataset, batch_size=8, shuffle=False,
    collate_fn=ocollate_fn, num_workers=4,
)

# Finetune: stroke only.
ft_dataset = COHWRDataset(
    DATA_ROOT, eval_df, tokenizer,
    img_feat=img_feat, aux_feat=[], cache_dir="val_cache3",
)
ft_loader = DataLoader(
    ft_dataset, batch_size=64, shuffle=False,
    collate_fn=ocollate_fn, num_workers=4,
)

In [4]:
# ------------------------------------------------------------------
# Model builders (architecture only; weights loaded per checkpoint)
# ------------------------------------------------------------------
def new_online_model():
    return JointMultiModel(
        trans_enc_d_model=256, trans_enc_nhead=8, trans_enc_layers=3, trans_enc_ff_dim=1024,
        tokenizer=tokenizer,
        trans_dec_d_model=256, trans_dec_nhead=8, trans_dec_layers=3, trans_dec_n_positions=512,
        freeze_decoder=False,
        encoder_path="partials/best_transformer_encoder_uhwr_icdar.pt",
        decoder_path="partials/best_transformer_decoder_uhwr_icdar.pt",
        cnn_encoder_path="partials/best_cnn_encoder_uhwr_icdar.pt",
        ctc_head_path="partials/best_ctc_head_uhwr_icdar.pt",
        img_feat=img_feat, aux_feat=aux_feat,
    ).to(device)


def new_finetune_model():
    return JointModel(
        trans_enc_d_model=256, trans_enc_nhead=8, trans_enc_layers=3, trans_enc_ff_dim=1024,
        tokenizer=tokenizer,
        trans_dec_d_model=256, trans_dec_nhead=8, trans_dec_layers=3, trans_dec_n_positions=512,
        freeze_decoder=False,
        decoder_path="decoder_pretrain_tokenizer_bos_eos/checkpoint-32452",
    ).to(device)

In [5]:
# ------------------------------------------------------------------
# Best-checkpoint selection: scan a window of epoch_*.pt, pick lowest CER.
# Window = the last SCAN_LAST_FRACTION of epochs, excluding the final
# SCAN_EXCLUDE_TAIL epochs (the no-improvement early-stopping tail; best.pt
# is the last-improvement checkpoint ~11th from the end). best.pt itself is
# NOT used and is often not the lowest-CER epoch.
# Short runs (e.g. finetune) fall back to just the last fraction so the
# window is never empty.
# ------------------------------------------------------------------
def list_epoch_ckpts(run_dir):
    files = [f for f in os.listdir(run_dir) if f.startswith("epoch_") and f.endswith(".pt")]
    files.sort(key=lambda f: int(f.split("_")[1].split(".")[0]))
    return files


def epoch_num(fname):
    return int(os.path.basename(fname).split("_")[1].split(".")[0])


def epochs_to_scan(files):
    n = len(files)
    start = int(n * (1.0 - SCAN_LAST_FRACTION))  # keep the last fraction
    end = n - SCAN_EXCLUDE_TAIL                   # drop the final tail epochs
    window = files[start:end]
    if not window:  # short run: relax the tail cut, keep the last fraction
        window = files[start:] or files
        print(f"  [warn] scan window empty after excluding last {SCAN_EXCLUDE_TAIL}; "
              f"falling back to {len(window)} ckpts")
    return window


def select_best_checkpoint(model, run_dir, loader, eval_fn, tag, seed, **eval_kwargs):
    os.environ["ABLATION_TYPE"] = "none"  # always full model for selection
    candidates = epochs_to_scan(list_epoch_ckpts(run_dir))
    print(f"[{tag} seed {seed}] scanning {len(candidates)} ckpts: "
          f"{candidates[0]} .. {candidates[-1]}")
    scores = []
    best_path, best_cer = None, float("inf")
    for f in candidates:
        path = os.path.join(run_dir, f)
        model.load_state_dict(torch.load(path, map_location=device))
        _, cer = eval_fn(
            model, loader, tokenizer, device, None,
            decode_mode=DECODE_MODE, loss_calc=False, **eval_kwargs,
        )
        cer_pct = cer * 100
        scores.append({"epoch": epoch_num(f), "checkpoint": f, "cer_percent": round(cer_pct, 4)})
        print(f"[{tag} seed {seed}] {f}: CER {cer_pct:.2f}%")
        if cer_pct < best_cer:
            best_path, best_cer = path, cer_pct
    scan_df = pd.DataFrame(scores).sort_values("epoch").reset_index(drop=True)
    scan_csv = f"cer_camera_ready_epoch_scan_{tag}_seed_{seed}.csv"
    scan_df.to_csv(scan_csv, index=False)
    print(f"  -> BEST {tag} seed {seed}: {os.path.basename(best_path)} @ {best_cer:.2f}%  (scan -> {scan_csv})")
    return best_path, best_cer, scan_df

In [6]:
# ------------------------------------------------------------------
# ONLINE: per seed -> select best checkpoint, then run aux ablations
#   none       -> Full (all channels)
#   img_stroke -> Ink only (zero all aux)
#   <aux feat> -> Ink + that one channel
# ------------------------------------------------------------------
ABLATIONS = ["none", "img_stroke"] + aux_feat


def pretty(ab):
    if ab == "none":
        return "Full"
    if ab == "img_stroke":
        return "Ink only"
    return f"Ink + {ab}"


online_model = new_online_model()
online_ablation_scores = {}   # seed -> {ablation: cer%}
online_selected = {}          # seed -> (epoch, full_cer)

for seed in SEEDS:
    run_dir = os.path.join(ONLINE_RUNS, f"seed_{seed}")
    best_path, full_cer, _ = select_best_checkpoint(
        online_model, run_dir, online_loader, evaluate_online, "online", seed,
    )
    online_selected[seed] = (epoch_num(best_path), full_cer)

    # Reload chosen checkpoint and run ablations (Full already known from scan).
    online_model.load_state_dict(torch.load(best_path, map_location=device))
    seed_scores = {"none": full_cer}
    for ab in ABLATIONS:
        if ab == "none":
            continue
        os.environ["ABLATION_TYPE"] = ab
        _, cer = evaluate_online(
            online_model, online_loader, tokenizer, device, None,
            decode_mode=DECODE_MODE, loss_calc=False,
        )
        seed_scores[ab] = cer * 100
        print(f"[online seed {seed}] {pretty(ab)}: {cer*100:.2f}%")
    os.environ["ABLATION_TYPE"] = "none"
    online_ablation_scores[seed] = seed_scores

Loaded pretrained original CNN.
Created fused encoder | Fusion: adaptive | Aux branches: 11
Loaded Transformer encoder weights from partials/best_transformer_encoder_uhwr_icdar.pt
Loaded Transformer decoder weights from partials/best_transformer_decoder_uhwr_icdar.pt
Loaded CTC head weights from partials/best_ctc_head_uhwr_icdar.pt
[online seed 42] scanning 7 ckpts: epoch_23.pt .. epoch_29.pt


Evaluating: 100%|██████████| 32/32 [00:51<00:00,  1.60s/it]


[online seed 42] epoch_23.pt: CER 3.73%


Evaluating: 100%|██████████| 32/32 [00:52<00:00,  1.63s/it]


[online seed 42] epoch_24.pt: CER 3.80%


...

[online seed 2024] Ink + img_pressure: 4.21%


Evaluating: 100%|██████████| 32/32 [00:48<00:00,  1.53s/it]


[online seed 2024] Ink + img_x_tilt: 4.58%


Evaluating: 100%|██████████| 32/32 [00:44<00:00,  1.41s/it]

[online seed 2024] Ink + img_y_tilt: 4.60%


In [7]:
# ------------------------------------------------------------------
# FINETUNE: per seed -> select best checkpoint (stroke only)
# ------------------------------------------------------------------
ft_model = new_finetune_model()
ft_scores = {}      # seed -> cer%
ft_selected = {}    # seed -> epoch

for seed in SEEDS:
    run_dir = os.path.join(FINETUNE_RUNS, f"seed_{seed}")
    best_path, best_cer, _ = select_best_checkpoint(
        ft_model, run_dir, ft_loader, evaluate_ft, "finetune", seed, data="img_stroke",
    )
    ft_scores[seed] = best_cer
    ft_selected[seed] = epoch_num(best_path)

  [warn] scan window empty after excluding last 9; falling back to 6 ckpts
[finetune seed 42] scanning 6 ckpts: epoch_7.pt .. epoch_12.pt


Evaluating: 100%|██████████| 4/4 [00:40<00:00, 10.23s/it]


[finetune seed 42] epoch_7.pt: CER 4.29%


Evaluating: 100%|██████████| 4/4 [00:35<00:00,  8.88s/it]


[finetune seed 42] epoch_8.pt: CER 4.27%


...

[finetune seed 2024] epoch_13.pt: CER 4.61%


Evaluating: 100%|██████████| 4/4 [00:37<00:00,  9.33s/it]


[finetune seed 2024] epoch_14.pt: CER 4.68%


Evaluating: 100%|██████████| 4/4 [00:38<00:00,  9.52s/it]

[finetune seed 2024] epoch_15.pt: CER 4.58%
  -> BEST finetune seed 2024: epoch_9.pt @ 4.16%  (scan -> cer_camera_ready_epoch_scan_finetune_seed_2024.csv)


In [8]:
# ------------------------------------------------------------------
# Score pairs: online (Full) vs finetune, per seed + mean
# ------------------------------------------------------------------
pairs = pd.DataFrame({
    "seed": [str(s) for s in SEEDS],
    "online_epoch": [online_selected[s][0] for s in SEEDS],
    "online_full_cer": [online_selected[s][1] for s in SEEDS],
    "finetune_epoch": [ft_selected[s] for s in SEEDS],
    "finetune_cer": [ft_scores[s] for s in SEEDS],
})
mean_row = {
    "seed": "mean",
    "online_epoch": "",
    "online_full_cer": pairs["online_full_cer"].mean(),
    "finetune_epoch": "",
    "finetune_cer": pairs["finetune_cer"].mean(),
}
pairs = pd.concat([pairs, pd.DataFrame([mean_row])], ignore_index=True)
pairs.to_csv("cer_camera_ready_pairs.csv", index=False)
pairs.round(4)

,seed,online_epoch,online_full_cer,finetune_epoch,finetune_cer
0,42,28,3.6716,12,4.1990
1,123,14,3.7017,9,4.1670
2,456,17,3.6670,11,4.3615
3,789,36,3.6028,11,4.1619
4,2024,18,3.6334,9,4.1551
5,mean,,3.6553,,4.2089


In [9]:
# ------------------------------------------------------------------
# Online aux ablation table: per seed + mean
# ------------------------------------------------------------------
abl = pd.DataFrame(
    {f"seed_{seed}": {pretty(ab): online_ablation_scores[seed][ab] for ab in ABLATIONS}
     for seed in SEEDS}
)
abl = abl.reindex([pretty(ab) for ab in ABLATIONS])
abl["mean"] = abl.mean(axis=1)
abl.to_csv("cer_camera_ready_ablations.csv")
abl.round(4)

,seed_42,seed_123,seed_456,seed_789,seed_2024,mean
Full,3.6716,3.7017,3.6670,3.6028,3.6334,3.6553
Ink only,5.0258,5.5274,5.4293,5.1836,5.3722,5.3076
Ink + img_dx,4.6685,4.6655,4.4848,4.5709,4.3573,4.5494
Ink + img_dy,4.8978,4.8941,4.6198,4.4255,4.4265,4.6528
Ink + img_sin_theta,4.7909,5.1713,5.2592,4.9280,5.1033,5.0505
Ink + img_cos_theta,4.7416,5.0293,4.7012,4.7234,4.5790,4.7549
Ink + img_curvature,4.5188,4.6253,4.2922,4.6597,4.5359,4.5264
Ink + img_speed,4.7285,4.6930,4.5281,4.6549,4.6165,4.6442
Ink + img_acceleration,4.7882,5.3135,5.3418,5.0436,4.9851,5.0944
Ink + img_time_norm,4.5703,4.6734,4.5869,4.4242,4.5272,4.5564
